In [ ]:
import numpy as np
from scipy.interpolate import RectBivariateSpline
from pyuvdata import UVBeam

def upsample_uvbeam_azza(
    beam: UVBeam,
    upsample_factor_za: int = 2,
    upsample_factor_az: int = 2,
    kx: int = 3,
    ky: int = 3,
) -> UVBeam:
    """
    Upsample a UVBeam defined on an (az, za) pixel grid to a finer grid.

    Parameters
    ----------
    beam : UVBeam
        Input beam with pixel_coordinate_system == 'az_za'.
        Assumes data_array shape: (Naxes_vec, Nfeeds, Nfreqs, Nza, Naz).
    upsample_factor_za : int
        Factor by which to increase the number of ZA samples.
    upsample_factor_az : int
        Factor by which to increase the number of AZ samples.
    kx, ky : int
        Spline degrees in ZA and AZ for RectBivariateSpline (1–5, typically 1–3).

    Returns
    -------
    beam_fine : UVBeam
        New beam with finer az/za grids and interpolated data_array.
    """
    if beam.pixel_coordinate_system != "az_za":
        raise ValueError("upsample_uvbeam_azza expects pixel_coordinate_system == 'az_za'")

    # Original grids
    az_old = beam.axis1_array  # (Naz,) azimuth (radians)
    za_old = beam.axis2_array  # (Nza,) zenith angle (radians)

    Naz_old = az_old.size
    Nza_old = za_old.size

    # Sanity: RectBivariateSpline requires strictly increasing axes
    if not (np.all(np.diff(za_old) > 0) and np.all(np.diff(az_old) > 0)):
        raise ValueError("za_old and az_old must be strictly increasing for RectBivariateSpline.")

    # New finer grids: same min/max, more samples
    Naz_new = Naz_old * upsample_factor_az
    Nza_new = Nza_old * upsample_factor_za

    az_new = np.linspace(az_old[0], az_old[-1], Naz_new)
    za_new = np.linspace(za_old[0], za_old[-1], Nza_new)

    # Prepare new beam and data array
    beam_fine = beam.copy()
    new_data = np.empty(
        (beam.Naxes_vec, beam.Nfeeds, beam.Nfreqs, Nza_new, Naz_new),
        dtype=beam.data_array.dtype,
    )

    # Loop over (vec, feed, freq) and spline-interpolate each 2D slice
    for ivec in range(beam.Naxes_vec):
        for ifeed in range(beam.Nfeeds):
            for ifreq in range(beam.Nfreqs):
                # slice_2d: (Nza_old, Naz_old)
                slice_2d = beam.data_array[ivec, ifeed, ifreq, :, :]

                # Real and imaginary parts separately
                spline_real = RectBivariateSpline(
                    za_old, az_old, slice_2d.real,
                    kx=kx, ky=ky
                )
                spline_imag = RectBivariateSpline(
                    za_old, az_old, slice_2d.imag,
                    kx=kx, ky=ky
                )

                # Evaluate on the finer grid: returns (Nza_new, Naz_new)
                vals_real = spline_real(za_new, az_new)
                vals_imag = spline_imag(za_new, az_new)

                new_slice = vals_real + 1j * vals_imag
                new_data[ivec, ifeed, ifreq, :, :] = new_slice.astype(beam.data_array.dtype)

    # Update beam metadata
    beam_fine.axis1_array = az_new
    beam_fine.axis2_array = za_new
    beam_fine.Naxes1 = Naz_new
    beam_fine.Naxes2 = Nza_new

    # For pixelized beams this is often used; adjust if your version uses a different attribute
    if hasattr(beam_fine, "Npixels"):
        beam_fine.Npixels = Naz_new * Nza_new

    beam_fine.data_array = new_data

    beam_fine.history += (
        f"\nUpsampled az/za grid by factors "
        f"(za x az) = ({upsample_factor_za} x {upsample_factor_az}) "
        f"using RectBivariateSpline(kx={kx}, ky={ky})."
    )

    return beam_fine


In [ ]:
import string
from pyuvdata import UVBeam

def airy_rotated_to_healpix_efield(
    beam_rot: UVBeam,
    nside: int | None = None,
    outfilename: str | None = None,
    clobber: bool = True,
) -> UVBeam:
    """
    Convert a rotated Airy UVBeam (efield, az_za) to a HEALPix efield UVBeam.

    Parameters
    ----------
    beam_rot : UVBeam
        Rotated beam, must have pixel_coordinate_system == 'az_za'
        and beam_type == 'efield'.
    nside : int or None
        Healpix nside. If None, pyuvdata will choose a suitable nside
        based on your az/za resolution.
    outfilename : str or None
        If given, write a beamfits file to this path.
    clobber : bool
        Overwrite existing file if True.

    Returns
    -------
    hpx_beam : UVBeam
        New UVBeam in HEALPix, efield format.
    """
    uvb = beam_rot.copy()

    # Sanity checks
    if uvb.pixel_coordinate_system != "az_za":
        raise ValueError("Expected az_za beam as input.")
    if uvb.beam_type != "efield":
        raise ValueError("Expected efield beam as input.")

    # Required interpolation scheme for az_za → healpix
    uvb.interpolation_function = "az_za_simple"

    # Convert to healpix efield (no 'order' kwarg in this pyuvdata version)
    if nside is None:
        hpx_beam = uvb.to_healpix(inplace=False)
    else:
        hpx_beam = uvb.to_healpix(nside=nside, inplace=False)

    # Just to be explicit:
    assert hpx_beam.beam_type == "efield"
    assert hpx_beam.pixel_coordinate_system == "healpix"

    # --- Clean history so FITS (ASCII-only) is happy ---
    def force_ascii(s: str) -> str:
        return "".join(ch if ch in string.printable else "?" for ch in s)

    hpx_beam.history = force_ascii(hpx_beam.history)
    hpx_beam.history += (
        "\nConverted rotated Airy beam from az_za to healpix "
        f"(beam_type=efield, nside={getattr(hpx_beam, 'nside', 'unknown')})."
    )
    hpx_beam.history = force_ascii(hpx_beam.history)

    # Optionally write out
    if outfilename is not None:
        hpx_beam.write_beamfits(outfilename, clobber=clobber)

    return hpx_beam


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyuvdata import UVBeam
from scipy.special import j1, j0
import gc
from copy import deepcopy

def create_airy_uvbeam(
    diameter: float,
    freq_array: np.ndarray,
    az_array: np.ndarray = None,
    za_array: np.ndarray = None,
    telescope_name: str = 'HERA',
    feed_name: str = 'airy',
    below_horizon: str = 'exponential',
    suppression_floor_db: float = -40.0,
    decay_rate_db_per_deg: float = 0.4,
    decay_start_za_deg: float = 90.0,
) -> UVBeam:
    """
    Create an Airy beam pattern and store in UVBeam format.
    
    Parameters
    ----------
    diameter : float
        Dish diameter in meters
    freq_array : np.ndarray
        Frequency array in Hz
    decay_rate_db_per_deg : float
        Decay rate in dB per degree past decay_start_za_deg
    decay_start_za_deg : float
        Zenith angle (degrees) where exponential decay begins.
        Default 90.0 (horizon). Use smaller values to start decay earlier.
    """

    if az_array is None:
        az_array = np.deg2rad(np.arange(0, 360, 1).astype(float))
    if za_array is None:
        za_array = np.deg2rad(np.arange(0, 181, 1).astype(float))

    freq_array = np.asarray(freq_array, dtype=float)

    c = 299792458.0
    n_freq = len(freq_array)
    n_az = len(az_array)
    n_za = len(za_array)

    k_array = 2 * np.pi * freq_array / c
    za_mesh, k_mesh = np.meshgrid(za_array, k_array, indexing='ij')
    x_vals = (diameter / 2.0) * np.sin(za_mesh) * k_mesh
#   print("x_vals :", x_vals.shape, x_vals)

    with np.errstate(divide='ignore', invalid='ignore'):
        airy_pattern = np.where(np.abs(x_vals) < 1e-10, 1.0, 2.0 * j1(x_vals) / x_vals)
    #   print("airy_pattern initial :", airy_pattern.shape, airy_pattern)

    # Exponential decay starting at decay_start_za_deg
    if below_horizon == 'exponential':
        for i_za, za in enumerate(za_array):
            za_deg = np.rad2deg(za)
            if za_deg > decay_start_za_deg:
                degrees_past_start = za_deg - decay_start_za_deg
                suppression_db = -decay_rate_db_per_deg * degrees_past_start
                suppression_db = max(suppression_db, suppression_floor_db)
                attenuation = 10 ** (suppression_db / 20)
                airy_pattern[i_za, :] *= attenuation

    efield_component = airy_pattern / np.sqrt(2)

    data_array = np.zeros((2, 2, n_freq, n_za, n_az), dtype=np.complex128)
    efield_transposed = efield_component.T

    for i_az in range(n_az):
        data_array[0, 0, :, :, i_az] = efield_transposed
        data_array[0, 1, :, :, i_az] = efield_transposed
        data_array[1, 0, :, :, i_az] = efield_transposed
        data_array[1, 1, :, :, i_az] = efield_transposed

    basis_vector_array = np.zeros((2, 2, n_za, n_az))
    basis_vector_array[0, 0, :, :] = 1.0
    basis_vector_array[1, 1, :, :] = 1.0

    beam = UVBeam()

    beam.pixel_coordinate_system = 'az_za'
    beam.beam_type = 'efield'
    beam.data_normalization = 'peak'
    beam.antenna_type = 'simple'

    beam.telescope_name = telescope_name
    beam.feed_name = feed_name
    beam.feed_version = '1.0'
    beam.model_name = f'Airy disk (D={diameter}m)'
    beam.model_version = '1.0'
    beam.history = (f'Analytic Airy beam D={diameter}m. '
                    f'Exponential decay {decay_rate_db_per_deg} dB/deg starting at ZA={decay_start_za_deg} deg.')

    beam.Naxes_vec = 2
    beam.Ncomponents_vec = 2
    beam.Nfeeds = 2
    beam.Nfreqs = n_freq
    beam.Naxes1 = n_az
    beam.Naxes2 = n_za

    beam.axis1_array = az_array
    beam.axis2_array = za_array
    beam.freq_array = freq_array

    beam.feed_array = np.array(['x', 'y'])
    beam.feed_angle = np.array([np.pi/2, 0.0])

    beam.data_array = data_array
    beam.basis_vector_array = basis_vector_array
    beam.bandpass_array = np.ones(n_freq, dtype=float)

    beam.check()

    return beam


# ============================================
# PARAMETERS - CHANGE THESE AS NEEDED
# ============================================
decay_rate = 0.3           # dB per degree
decay_start_za = 70.0      # zenith angle where decay starts (degrees)
diameter = 7.0            # dish diameter (meters)
freq_array = np.linspace(45e6, 250e6, 206)

# ============================================
# CREATE BEAM
# ============================================
print(f"Creating Airy beam:")
print(f"  Diameter: {diameter} m")
print(f"  Decay rate: {decay_rate} dB/deg")
print(f"  Decay starts at: ZA = {decay_start_za} deg")

beam = create_airy_uvbeam(
    diameter=diameter,
    freq_array=freq_array,
    telescope_name='HERA',
    feed_name='airy_exponential',
    below_horizon='exponential',
    decay_rate_db_per_deg=decay_rate,
    decay_start_za_deg=decay_start_za,
)

unrot_beam = beam.copy()  # keep unrotated copy for comparison

beam_fine = upsample_uvbeam_azza(
    beam,
    upsample_factor_za=2,   # or 3, etc.
    upsample_factor_az=2,
    kx=3,
    ky=3,
)

# ============================================
# ROTATE BEAM TO POINTING DIRECTION (optional)
# ============================================
rotate = False  # Set to True to rotate beam
if rotate :
    print("Rotating beam")
    # alt0_deg=45.0 
    # az0_deg=0.0
    # R = rotation_from_zenith_to_altaz(alt_deg=alt0_deg, az_deg=az0_deg)
    # beam_rotated = rotate_uvbeam_with_matrix(beam, R)
    # # rotate_axisymmetric_beam_to_point(beam, alt0_deg=alt0_deg, az0_deg=az0_deg) 
    # beam = beam_rotated
    
    # Suppose `beam` is your Airy UVBeam (zenith-pointing)
    theta0_deg = 2.0   # tilt away from zenith
    phi0_deg   = 270.0   # rotate in azimuth

    fine_rot_beam = rotate_uvbeam_pattern(beam_fine, theta0_deg, phi0_deg, degrees=True)

    # Check: where is the new peak?
    # (rough check by brute-force argmax on one freq, one feed/component)
    freq_idx = 100
    slice_power = np.abs(fine_rot_beam.data_array[0, 0, freq_idx])**2
    iz, iaz = np.unravel_index(np.argmax(slice_power), slice_power.shape)
    print("Peak at za' [deg], az' [deg] =",
        np.rad2deg(fine_rot_beam.axis2_array[iz]),
        np.rad2deg(fine_rot_beam.axis1_array[iaz]))
        
    # Step 3: downsample rotated beam back to original grid
    beam_rot_coarse = resample_uvbeam_to_template(
        fine_rot_beam,
        template_beam=beam,
        kx=3,
        ky=3,
    )
    
    beam = deepcopy(beam_rot_coarse)



# Save with parameters in filename
save_out = "/home/herastore02-1/HERA_Validation_rchandra/"
if rotate:
    filename = save_out + f'airy_beam_{diameter}m_decay_{decay_rate}dBdeg_start_{decay_start_za}deg_rttd_za_az_{theta0_deg}_{phi0_deg}.fits'
else:
    filename = save_out + f'airy_beam_{diameter}m_decay_{decay_rate}dBdeg_start_{decay_start_za}deg.fits'
beam.write_beamfits(filename, clobber=True)
print(f"\nSaved: {filename}")

if not rotate:
    # Choose nside ~ 64 for ~1° resolution; adjust if you want finer/coarser
    healpix_filename = save_out + f'airy_beam_{diameter}m_decay_{decay_rate}dBdeg_start_{decay_start_za}deg_healpix.fits'

    beam_hpx = airy_rotated_to_healpix_efield(
        beam,
        nside=64,
        outfilename=healpix_filename,
        clobber=True,
    )

    print("Wrote HEALPix efield beam to:", healpix_filename)
    print("beam_type:", beam_hpx.beam_type)
    print("pixel_coordinate_system:", beam_hpx.pixel_coordinate_system)
    print("nside:", getattr(beam_hpx, "nside", None))



# ============================================
# CHECK BASIC METADATA
# ============================================
print("\n" + "=" * 60)
print("BEAM METADATA")
print("=" * 60)
print(f"Beam is valid:       {beam.check()}")
print(f"Telescope Name:      {beam.telescope_name}")
print(f"Feed Name:           {beam.feed_name}")
print(f"Model Name:          {beam.model_name}")
print(f"Beam type:           {beam.beam_type}")
print(f"Coordinate system:   {beam.pixel_coordinate_system}")
print(f"Data normalization:  {beam.data_normalization}")
print(f"Data shape:          {beam.data_array.shape}")
print(f"Data dtype:          {beam.data_array.dtype}")
print(f"Freq range:          {beam.freq_array.min()/1e6:.1f} - {beam.freq_array.max()/1e6:.1f} MHz")
print(f"Num frequencies:     {beam.Nfreqs}")
print(f"Naxes1 (azimuth):    {beam.Naxes1}")
print(f"Naxes2 (zenith):     {beam.Naxes2}")
print(f"Feed array:          {beam.feed_array}")
print(f"Feed angle:          {beam.feed_angle}")
print(f"NaNs in beam:        {np.isnan(beam.data_array).sum()}")
print(f"Infs in beam:        {np.isinf(beam.data_array).sum()}")
print(f"History:             {beam.history}")


# ============================================
# PLOT BEAM PATTERN
# ============================================
freq_idx = np.argmin(np.abs(freq_array - 150e6))
actual_freq = freq_array[freq_idx] / 1e6
za_deg = np.rad2deg(beam.axis2_array)

e_theta = beam.data_array[0, 0, freq_idx, :, 0]
# print("e_theta :", e_theta.shape, e_theta)
e_phi = beam.data_array[1, 0, freq_idx, :, 0]
power = np.abs(e_theta)**2 + np.abs(e_phi)**2
power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Airy beam ({decay_rate} dB/deg decay)')
ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
            label=f'Decay start (ZA={decay_start_za}°)')
ax.axhline(-40, color='gray', linestyle=':', alpha=0.7, label='Floor (-40 dB)')

ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
ax.set_ylabel('Normalized Power (dB)', fontsize=12)
ax.set_title(f'Airy Beam ({diameter}m, {decay_rate} dB/deg decay from ZA={decay_start_za}°) at {actual_freq:.1f} MHz',
            fontsize=14)
ax.set_xlim(0, 180)
ax.set_ylim(-60, 5)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

plt.tight_layout()
plot_filename = f'airy_beam_{diameter}m_decay_{decay_rate}dBdeg_start_{decay_start_za}deg.png'
plt.savefig(plot_filename, dpi=150)
plt.show()

if rotate:
    # ============================================
    # PLOT BEAM PATTERN
    # ============================================
    freq_idx = np.argmin(np.abs(freq_array - 150e6))
    actual_freq = freq_array[freq_idx] / 1e6
    za_deg = np.rad2deg(fine_rot_beam.axis2_array)

    e_theta = fine_rot_beam.data_array[0, 0, freq_idx, :, 0]
    # print("e_theta :", e_theta.shape, e_theta)
    e_phi = fine_rot_beam.data_array[1, 0, freq_idx, :, 0]
    power = np.abs(e_theta)**2 + np.abs(e_phi)**2
    power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Airy beam ({decay_rate} dB/deg decay)')
    ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
                label=f'Decay start (ZA={decay_start_za}°)')
    ax.axhline(-40, color='gray', linestyle=':', alpha=0.7, label='Floor (-40 dB)')

    ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
    ax.set_ylabel('Normalized Power (dB)', fontsize=12)
    ax.set_title(f'Airy Beam ({diameter}m, {decay_rate} dB/deg decay from ZA={decay_start_za}°) at {actual_freq:.1f} MHz',
                fontsize=14)
    ax.set_xlim(0, 180)
    ax.set_ylim(-60, 5)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
    ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
    ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

    plt.tight_layout()
    plot_filename = f'airy_beam_{diameter}m_decay_{decay_rate}dBdeg_start_{decay_start_za}deg.png'
    plt.savefig(plot_filename, dpi=150)
    plt.show()

# print(f"\nPlot saved: {plot_filename}")
print("Done!")

In [ ]:
import numpy as np
from pyuvdata import UVBeam


def create_gaussian_uvbeam(
    sigma_deg: float = None,
    diameter: float = None,
    freq_array: np.ndarray = None,
    az_array: np.ndarray = None,
    za_array: np.ndarray = None,
    telescope_name: str = "HERA",
    feed_name: str = "gaussian",
    below_horizon: str = "exponential",
    suppression_floor_db: float = -40.0,
    decay_rate_db_per_deg: float = 0.4,
    decay_start_za_deg: float = 90.0,
) -> UVBeam:
    """
    Create a Gaussian beam pattern and store in UVBeam format.

    The beam voltage pattern is  E(za) = exp( -za^2 / (2 sigma^2) ).

    Sigma can be specified in two ways (supply exactly one):

    1. **sigma_deg** – a fixed angular width (degrees), constant across
       frequency.  Useful for quick tests.
    2. **diameter** – dish diameter in metres.  Sigma is then derived
       from the diffraction limit at each frequency:

           FWHM(f) = 1.02 * lambda / D          (radians)
           sigma(f) = FWHM / (2 sqrt(2 ln 2))   (radians)

       so the beam narrows at higher frequencies, as expected.

    Parameters
    ----------
    sigma_deg : float or None
        Fixed Gaussian sigma in degrees (frequency-independent).
    diameter : float or None
        Dish diameter in metres (gives frequency-dependent sigma).
    freq_array : np.ndarray
        Frequencies in Hz.
    az_array, za_array : np.ndarray or None
        Azimuth / zenith-angle grids in radians.  Defaults to 1-deg steps.
    below_horizon : str
        'exponential' to apply decaying envelope past `decay_start_za_deg`,
        or 'none' to leave the Gaussian tail untouched.
    decay_rate_db_per_deg : float
        Decay rate in dB per degree past `decay_start_za_deg`.
    decay_start_za_deg : float
        ZA (degrees) where exponential suppression begins.
    suppression_floor_db : float
        Minimum suppression in dB (floor).

    Returns
    -------
    beam : UVBeam
        E-field UVBeam on an az_za pixel grid.
    """
    # ---- Input validation ------------------------------------------------
    if (sigma_deg is None) == (diameter is None):
        raise ValueError("Supply exactly one of `sigma_deg` or `diameter`.")
    if freq_array is None:
        raise ValueError("`freq_array` is required.")

    if az_array is None:
        az_array = np.deg2rad(np.arange(0, 360, 1).astype(float))
    if za_array is None:
        za_array = np.deg2rad(np.arange(0, 181, 1).astype(float))

    freq_array = np.asarray(freq_array, dtype=float)
    c = 299792458.0
    n_freq = len(freq_array)
    n_az = len(az_array)
    n_za = len(za_array)

    # ---- Build sigma array (one per frequency) ---------------------------
    if sigma_deg is not None:
        # Constant sigma across frequency
        sigma_rad = np.full(n_freq, np.deg2rad(sigma_deg))
    else:
        # Diffraction-limited:  FWHM = 1.02 * lambda / D
        wavelengths = c / freq_array
        fwhm_rad = 1.02 * wavelengths / diameter
        sigma_rad = fwhm_rad / (2.0 * np.sqrt(2.0 * np.log(2.0)))

    # ---- Gaussian pattern on (za, freq) grid ----------------------------
    # za_mesh shape: (n_za, n_freq),  sigma_mesh same
    za_mesh, sigma_mesh = np.meshgrid(za_array, sigma_rad, indexing="ij")
    gaussian_pattern = np.exp(-0.5 * (za_mesh / sigma_mesh) ** 2)  # (n_za, n_freq)

    # ---- Below-horizon suppression (identical to Airy version) -----------
    if below_horizon == "exponential":
        for i_za, za in enumerate(za_array):
            za_deg = np.rad2deg(za)
            if za_deg > decay_start_za_deg:
                degrees_past = za_deg - decay_start_za_deg
                suppression_db = max(-decay_rate_db_per_deg * degrees_past,
                                     suppression_floor_db)
                gaussian_pattern[i_za, :] *= 10.0 ** (suppression_db / 20.0)
    if below_horizon != "exponential":
        print("below horizon suppression disabled ", below_horizon)

    # ---- Pack into UVBeam ------------------------------------------------
    efield_component = gaussian_pattern / np.sqrt(2)
    efield_transposed = efield_component.T  # (n_freq, n_za)

    data_array = np.zeros((2, 2, n_freq, n_za, n_az), dtype=np.complex128)
    for i_az in range(n_az):
        data_array[0, 0, :, :, i_az] = efield_transposed
        data_array[0, 1, :, :, i_az] = efield_transposed
        data_array[1, 0, :, :, i_az] = efield_transposed
        data_array[1, 1, :, :, i_az] = efield_transposed

    basis_vector_array = np.zeros((2, 2, n_za, n_az))
    basis_vector_array[0, 0, :, :] = 1.0
    basis_vector_array[1, 1, :, :] = 1.0

    beam = UVBeam()
    beam.pixel_coordinate_system = "az_za"
    beam.beam_type = "efield"
    beam.data_normalization = "peak"
    beam.antenna_type = "simple"

    beam.telescope_name = telescope_name
    beam.feed_name = feed_name
    beam.feed_version = "1.0"
    if sigma_deg is not None:
        beam.model_name = f"Gaussian (sigma={sigma_deg:.2f} deg, fixed)"
    else:
        beam.model_name = f"Gaussian (D={diameter}m, diffraction-limited)"
    beam.model_version = "1.0"

    sigma_info = (f"sigma_deg={sigma_deg}" if sigma_deg is not None
                  else f"D={diameter}m, sigma(150 MHz)="
                       f"{np.rad2deg(sigma_rad[np.argmin(np.abs(freq_array-150e6))]):.2f} deg")
    beam.history = (
        f"Analytic Gaussian beam ({sigma_info}). "
        f"Exponential decay {decay_rate_db_per_deg} dB/deg "
        f"starting at ZA={decay_start_za_deg} deg."
    )

    beam.Naxes_vec = 2
    beam.Ncomponents_vec = 2
    beam.Nfeeds = 2
    beam.Nfreqs = n_freq
    beam.Naxes1 = n_az
    beam.Naxes2 = n_za

    beam.axis1_array = az_array
    beam.axis2_array = za_array
    beam.freq_array = freq_array

    beam.feed_array = np.array(["x", "y"])
    beam.feed_angle = np.array([np.pi / 2, 0.0])

    beam.data_array = data_array
    beam.basis_vector_array = basis_vector_array
    beam.bandpass_array = np.ones(n_freq, dtype=float)

    beam.check()
    return beam

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# ============================================
# PARAMETERS - CHANGE THESE AS NEEDED
# ============================================
decay_rate = 0.3           # dB per degree
decay_start_za = 70.0      # zenith angle where decay starts (degrees)
diameter = 7.0            # dish diameter (meters) — used for freq-dependent sigma
sigma_deg = None           # set to e.g. 10.0 for fixed sigma; leave None to use diameter
freq_array = np.linspace(45e6, 250e6, 206)

# ============================================
# CREATE GAUSSIAN BEAM
# ============================================
if sigma_deg is not None:
    print(f"Creating Gaussian beam (fixed sigma):")
    print(f"  sigma: {sigma_deg} deg")
else:
    print(f"Creating Gaussian beam (diffraction-limited):")
    print(f"  Diameter: {diameter} m")
print(f"  Decay rate: {decay_rate} dB/deg")
print(f"  Decay starts at: ZA = {decay_start_za} deg")

gauss_beam = create_gaussian_uvbeam(
    sigma_deg=sigma_deg,
    diameter=diameter,
    freq_array=freq_array,
    telescope_name='HERA',
    feed_name='gaussian_exponential',
    below_horizon='exponential',
    decay_rate_db_per_deg=decay_rate,
    decay_start_za_deg=decay_start_za,
)

unrot_gauss_beam = gauss_beam.copy()  # keep unrotated copy for comparison

gauss_beam_fine = upsample_uvbeam_azza(
    gauss_beam,
    upsample_factor_za=2,
    upsample_factor_az=2,
    kx=3,
    ky=3,
)

# ============================================
# ROTATE BEAM TO POINTING DIRECTION (optional)
# ============================================
rotate_gauss = False  # Set to True to rotate beam
if rotate_gauss:
    print("Rotating Gaussian beam")

    theta0_deg = 2.0   # tilt away from zenith
    phi0_deg   = 270.0  # rotate in azimuth

    fine_rot_gauss_beam = rotate_uvbeam_pattern(gauss_beam_fine, theta0_deg, phi0_deg, degrees=True)

    # Check: where is the new peak?
    freq_idx = 100
    slice_power = np.abs(fine_rot_gauss_beam.data_array[0, 0, freq_idx])**2
    iz, iaz = np.unravel_index(np.argmax(slice_power), slice_power.shape)
    print("Peak at za' [deg], az' [deg] =",
        np.rad2deg(fine_rot_gauss_beam.axis2_array[iz]),
        np.rad2deg(fine_rot_gauss_beam.axis1_array[iaz]))

    # Downsample rotated beam back to original grid
    gauss_beam_rot_coarse = resample_uvbeam_to_template(
        fine_rot_gauss_beam,
        template_beam=gauss_beam,
        kx=3,
        ky=3,
    )

    gauss_beam = deepcopy(gauss_beam_rot_coarse)

# ============================================
# BUILD FILENAME TAG
# ============================================
if sigma_deg is not None:
    beam_tag = f"gaussian_beam_sigma_{sigma_deg}deg"
else:
    beam_tag = f"gaussian_beam_{diameter}m"
beam_tag += f"_decay_{decay_rate}dBdeg_start_{decay_start_za}deg"

# ============================================
# SAVE TO DISK
# ============================================
save = "ON" # Set to "OFF" to skip saving files
if save == "ON":
    save_out = "/home/herastore02-1/HERA_Validation_rchandra/"
    if rotate_gauss:
        filename = save_out + f'{beam_tag}_rttd_za_az_{theta0_deg}_{phi0_deg}.fits'
    else:
        filename = save_out + f'{beam_tag}.fits'
    gauss_beam.write_beamfits(filename, clobber=True)
    print(f"\nSaved: {filename}")

    if not rotate_gauss:
        healpix_filename = save_out + f'{beam_tag}_healpix.fits'

        gauss_beam_hpx = airy_rotated_to_healpix_efield(
            gauss_beam,
            nside=64,
            outfilename=healpix_filename,
            clobber=True,
        )

        print("Wrote HEALPix efield beam to:", healpix_filename)
        print("beam_type:", gauss_beam_hpx.beam_type)
        print("pixel_coordinate_system:", gauss_beam_hpx.pixel_coordinate_system)
        print("nside:", getattr(gauss_beam_hpx, "nside", None))

# ============================================
# CHECK BASIC METADATA
# ============================================
print("\n" + "=" * 60)
print("GAUSSIAN BEAM METADATA")
print("=" * 60)
print(f"Beam is valid:       {gauss_beam.check()}")
print(f"Telescope Name:      {gauss_beam.telescope_name}")
print(f"Feed Name:           {gauss_beam.feed_name}")
print(f"Model Name:          {gauss_beam.model_name}")
print(f"Beam type:           {gauss_beam.beam_type}")
print(f"Coordinate system:   {gauss_beam.pixel_coordinate_system}")
print(f"Data normalization:  {gauss_beam.data_normalization}")
print(f"Data shape:          {gauss_beam.data_array.shape}")
print(f"Data dtype:          {gauss_beam.data_array.dtype}")
print(f"Freq range:          {gauss_beam.freq_array.min()/1e6:.1f} - {gauss_beam.freq_array.max()/1e6:.1f} MHz")
print(f"Num frequencies:     {gauss_beam.Nfreqs}")
print(f"Naxes1 (azimuth):    {gauss_beam.Naxes1}")
print(f"Naxes2 (zenith):     {gauss_beam.Naxes2}")
print(f"Feed array:          {gauss_beam.feed_array}")
print(f"Feed angle:          {gauss_beam.feed_angle}")
print(f"NaNs in beam:        {np.isnan(gauss_beam.data_array).sum()}")
print(f"Infs in beam:        {np.isinf(gauss_beam.data_array).sum()}")
print(f"History:             {gauss_beam.history}")

# ============================================
# PLOT BEAM PATTERN
# ============================================
freq_idx = np.argmin(np.abs(freq_array - 150e6))
actual_freq = freq_array[freq_idx] / 1e6
za_deg = np.rad2deg(gauss_beam.axis2_array)

e_theta = gauss_beam.data_array[0, 0, freq_idx, :, 0]
e_phi = gauss_beam.data_array[1, 0, freq_idx, :, 0]
power = np.abs(e_theta)**2 + np.abs(e_phi)**2
power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Gaussian beam ({decay_rate} dB/deg decay)')
ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
            label=f'Decay start (ZA={decay_start_za}°)')
ax.axhline(-40, color='gray', linestyle=':', alpha=0.7, label='Floor (-40 dB)')

ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
ax.set_ylabel('Normalized Power (dB)', fontsize=12)
if sigma_deg is not None:
    ax.set_title(f'Gaussian Beam (σ={sigma_deg}°, {decay_rate} dB/deg decay from ZA={decay_start_za}°) at {actual_freq:.1f} MHz', fontsize=14)
else:
    ax.set_title(f'Gaussian Beam (D={diameter}m, {decay_rate} dB/deg decay from ZA={decay_start_za}°) at {actual_freq:.1f} MHz', fontsize=14)
ax.set_xlim(0, 180)
# ax.set_ylim(-60, 5)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

plt.tight_layout()
plot_filename = f'{beam_tag}.png'
plt.savefig(plot_filename, dpi=150)
plt.show()

if rotate_gauss:
    freq_idx = np.argmin(np.abs(freq_array - 150e6))
    actual_freq = freq_array[freq_idx] / 1e6
    za_deg = np.rad2deg(fine_rot_gauss_beam.axis2_array)

    e_theta = fine_rot_gauss_beam.data_array[0, 0, freq_idx, :, 0]
    e_phi = fine_rot_gauss_beam.data_array[1, 0, freq_idx, :, 0]
    power = np.abs(e_theta)**2 + np.abs(e_phi)**2
    power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Gaussian beam ({decay_rate} dB/deg decay)')
    ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
                label=f'Decay start (ZA={decay_start_za}°)')
    ax.axhline(-40, color='gray', linestyle=':', alpha=0.7, label='Floor (-40 dB)')

    ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
    ax.set_ylabel('Normalized Power (dB)', fontsize=12)
    ax.set_title(f'Rotated Gaussian Beam at {actual_freq:.1f} MHz', fontsize=14)
    ax.set_xlim(0, 180)
    ax.set_ylim(-60, 5)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
    ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
    ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

    plt.tight_layout()
    plot_filename = f'{beam_tag}_rotated.png'
    # plt.savefig(plot_filename, dpi=150)
    plt.show()

print("Done!")

# Analytic Form of the Gaussian Beam Generator



This cell summarizes the exact functional form implemented by `create_gaussian_uvbeam` in this notebook.



## 1. Core Gaussian E-field envelope



Let $\theta$ be zenith angle in radians, $\phi$ be azimuth, and $f$ be frequency. Before any optional rotation, the generated beam is azimuth-independent, so it depends only on $(\theta, f)$.



The scalar Gaussian voltage pattern is



$$

G_0(\theta, f) = \exp\!\left[-\frac{1}{2}\left(\frac{\theta}{\sigma(f)}\right)^2\right].

$$



The width $\sigma(f)$ is defined in one of two ways.



### Fixed-sigma mode



If `sigma_deg` is supplied, then the width is frequency-independent:



$$

\sigma(f) = \sigma_0 = \frac{\pi}{180}\,\sigma_{\rm deg}.

$$



### Diameter-based mode



If `diameter = D` is supplied instead, then the notebook uses a diffraction-limited FWHM:



$$

\lambda(f) = \frac{c}{f},

$$



$$

{\rm FWHM}(f) = 1.02\,\frac{\lambda(f)}{D} = 1.02\,\frac{c}{Df},

$$



and converts to Gaussian sigma via



$$

\sigma(f) = \frac{{\rm FWHM}(f)}{2\sqrt{2\ln 2}} = \frac{1.02\,c}{2\sqrt{2\ln 2}\,D\,f}.

$$



So in this mode the beam width scales as $\sigma(f) \propto f^{-1}$.



## 2. Below-horizon exponential suppression



The notebook then multiplies the Gaussian by a piecewise attenuation in amplitude, written in dB per degree beyond a chosen zenith angle.



Let



- $\theta_{\rm deg} = 180\theta/\pi$

- $\theta_{\rm start}$ be `decay_start_za_deg`

- $r$ be `decay_rate_db_per_deg`

- $s_{\rm floor}$ be `suppression_floor_db`



The suppression in dB is



$$

s_{\rm dB}(\theta) =

\begin{cases}

0, & \theta_{\rm deg} \le \theta_{\rm start}, \\

\max\!\left[-r\,(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}\right], & \theta_{\rm deg} > \theta_{\rm start}.

\end{cases}

$$



The corresponding multiplicative amplitude envelope is



$$

E(\theta) = 10^{s_{\rm dB}(\theta)/20}.

$$



Therefore the final scalar E-field amplitude generated by the function is



$$

G(\theta, f) = \exp\!\left[-\frac{1}{2}\left(\frac{\theta}{\sigma(f)}\right)^2\right]

\times

\begin{cases}

1, & \theta_{\rm deg} \le \theta_{\rm start}, \\

10^{\max[-r(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}]/20}, & \theta_{\rm deg} > \theta_{\rm start}.

\end{cases}

$$



This is the exact analytic form of the notebook beam envelope.



## 3. Jones matrix actually stored in the UVBeam



The notebook stores this as an `efield` beam and assigns the same real scalar amplitude to all four Jones-matrix entries, with an extra factor of $1/\sqrt{2}$:



$$

J_{ab}(\phi, \theta, f) = \frac{G(\theta,f)}{\sqrt{2}}, \qquad a,b \in \{1,2\}.

$$



Equivalently,



$$

J(\phi,\theta,f) = \frac{G(\theta,f)}{\sqrt{2}}

\begin{pmatrix}

1 & 1 \\

1 & 1

\end{pmatrix}.

$$



So, before any optional rotation:



- there is no intrinsic azimuthal dependence

- there is no phase term

- there are no oscillatory rings or sidelobes

- there is no additional frequency envelope beyond the $\sigma(f)$ scaling

- the bandpass is unity at all frequencies



Since $G(0,f)=1$, the beam is peak-normalized at zenith.



## 4. Power beam implied by the notebook plotting code



The plotting cell forms power using



$$

P(\theta,f) = |E_\theta|^2 + |E_\phi|^2.

$$



Because each component carries $G(\theta,f)/\sqrt{2}$, this becomes



$$

P(\theta,f) = G(\theta,f)^2.

$$



Hence the power beam is



$$

P(\theta, f) = \exp\!\left[-\left(\frac{\theta}{\sigma(f)}\right)^2\right]

\times

\begin{cases}

1, & \theta_{\rm deg} \le \theta_{\rm start}, \\

10^{\max[-r(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}]/10}, & \theta_{\rm deg} > \theta_{\rm start}.

\end{cases}

$$



## 5. Memo-style compact form



For quick reference, the notebook Gaussian beam is:



$$

\boxed{

G(\theta,f)=\exp\!\left[-\frac{1}{2}\left(\frac{\theta}{\sigma(f)}\right)^2\right]

\times

A(\theta)

}

$$



with



$$

A(\theta)=

\begin{cases}

1, & \theta_{\rm deg}\le \theta_{\rm start}, \\

10^{\max[-r(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}]/20}, & \theta_{\rm deg}>\theta_{\rm start},

\end{cases}

$$



and



$$

\sigma(f)=

\begin{cases}

\dfrac{\pi}{180}\,\sigma_{\rm deg}, & \text{fixed-sigma mode}, \\

\dfrac{1.02\,c}{2\sqrt{2\ln 2}\,D\,f}, & \text{diameter-based mode}.

\end{cases}

$$



The stored Jones beam is



$$

\boxed{

J(\phi,\theta,f)=\dfrac{G(\theta,f)}{\sqrt{2}}

\begin{pmatrix}

1 & 1 \\

1 & 1

\end{pmatrix}

}

$$



and the plotted power beam is simply



$$

\boxed{P(\theta,f)=G(\theta,f)^2.}

$$



## 6. Note on optional rotation



If `rotate_gauss = True`, the notebook later rotates the beam pattern geometrically on the sky. That does not change the intrinsic functional form above; it only remaps the same axisymmetric beam to a new pointing direction, which then introduces azimuth dependence through coordinate transformation rather than through a new envelope term.
